# Parks and conservation reserves: load and inspect

This notebook loads the PARKRES shapefile, checks its structure, and
prepares a point (latitude and longitude) for each park so it can be
used the same way as the facilities and shopping centres.

## Imports and configuration

`geopandas` reads the shapefile and handles its coordinate system.
`PARKRES_PATH` points at the `.shp` file; the other files in that folder
(`.shx`, `.dbf`, `.prj` and so on) are read automatically alongside it.

In [14]:
import geopandas as gpd
import pandas as pd
import numpy as np

PARKRES_PATH = "../../external_data/parks/ll_gda2020/esrishape/whole_of_dataset/victoria/CROWNLAND/PARKRES.shp"

pd.set_option("display.max_columns", None)

## Read the shapefile

`parks_raw` is the untouched data. Every cleaning step from here
produces a new object, following the same convention as the facilities
and shopping centres notebooks.

In [15]:
parks_raw = gpd.read_file(PARKRES_PATH)

### Coordinate reference system

Check which coordinate system the shapefile uses before doing anything
with the coordinates. Victoria's current standard is GDA2020, and this
should say so (EPSG:7844 for latitude/longitude)

In [16]:
print(parks_raw.crs)

EPSG:7844


### Size, columns and geometry type

`info()` lists every column, its data type and how many non-null values
it has, plus the total row count. The geometry type is checked
separately, since parks are expected to be polygons (or multipolygons
for parks in several pieces) and not points or lines.

In [17]:
parks_raw.info()
print(parks_raw.geometry.geom_type.value_counts())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 8963 entries, 0 to 8962
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   PRIMS_ID    8963 non-null   int64         
 1   NAME        8963 non-null   object        
 2   AREA_TYPE   8963 non-null   object        
 3   NAME_SHORT  8963 non-null   object        
 4   AREA_SRC    8963 non-null   object        
 5   MANAGER     8963 non-null   object        
 6   VEAC_REC    8888 non-null   object        
 7   VEAC_STUDY  8963 non-null   object        
 8   IUCN        8963 non-null   object        
 9   TOTAL_AREA  8963 non-null   float64       
 10  POLY_SRC    8963 non-null   object        
 11  AREATYPEID  8963 non-null   int64         
 12  SRCHNAME    8963 non-null   object        
 13  ESTAB_DATE  8434 non-null   datetime64[ms]
 14  LAST_MOD    8963 non-null   datetime64[ms]
 15  VERS_DATE   8963 non-null   datetime64[ms]
 16  HECTARES    8963

### Check for duplicates and nulls

Compare the number of rows against the number of unique geometries,
since a truly duplicated park would appear as two identical rows. Also
check every column for nulls, to see whether an ID or name column is
complete enough to rely on.

In [18]:
n_duplicate_geometries = parks_raw.duplicated(subset="geometry").sum()
print(f"Duplicate geometries: {n_duplicate_geometries}")

print(parks_raw.isna().sum())

Duplicate geometries: 9
PRIMS_ID        0
NAME            0
AREA_TYPE       0
NAME_SHORT      0
AREA_SRC        0
MANAGER         0
VEAC_REC       75
VEAC_STUDY      0
IUCN            0
TOTAL_AREA      0
POLY_SRC        0
AREATYPEID      0
SRCHNAME        0
ESTAB_DATE    529
LAST_MOD        0
VERS_DATE       0
HECTARES        0
AREASQM         0
geometry        0
dtype: int64


### Look at the duplicate geometries

Check whether the 9 rows with duplicate geometries also share a
`PRIMS_ID`. If the IDs differ, these may be two different records for
the same area (for example a reservation and a park layered on the same
land) rather than a data-entry duplicate.

In [19]:
duplicate_geometries = parks_raw[
    parks_raw.duplicated(subset="geometry", keep=False)
].sort_values("geometry", key=lambda col: col.astype(str))

duplicate_geometries[["PRIMS_ID", "NAME", "AREA_TYPE", "HECTARES"]]

,PRIMS_ID,NAME,AREA_TYPE,HECTARES
1546,5228,Great Otway National Park GORCAPA,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",136.83
8059,5228,Great Otway National Park GORCAPA,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",136.83
5813,4710,Great Otway National Park,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",108.37
5856,4710,Great Otway National Park,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",108.37
6680,5228,Great Otway National Park GORCAPA,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",7.78
3432,5228,Great Otway National Park GORCAPA,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",7.78
7567,5228,Great Otway National Park GORCAPA,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",4.54
6376,4710,Great Otway National Park,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",4.54
7743,3326,Arthurs Seat State Park,"STATE PARK - SCHEDULE 2B, NATIONAL PARKS ACT",0.07
5465,3326,Arthurs Seat State Park,"STATE PARK - SCHEDULE 2B, NATIONAL PARKS ACT",0.07


### Remove true duplicate parcels

A true duplicate is the same parcel entered twice: identical `PRIMS_ID`,
`NAME` and geometry. This is different from a park with several distinct
parcels, which correctly share a `PRIMS_ID` but have different shapes
and areas. `drop_duplicates` on the geometry column removes only the
former and leaves every distinct parcel in place.

In [20]:
n_before = len(parks_raw)
parks_raw = parks_raw.drop_duplicates(subset="geometry").reset_index(
    drop=True
)
n_after = len(parks_raw)

print(f"Removed {n_before - n_after} duplicate parcels")
print(f"Rows remaining: {n_after}")

Removed 9 duplicate parcels
Rows remaining: 8954


### Look at the area types

List the categories in `AREA_TYPE` and how many parks fall into each,
and the smallest and largest reserves by `HECTARES`. Some categories may
not be places the public visits (for example a narrow road reserve or a
remote reference area), and a very small `HECTARES` value can point to
a sliver polygon rather than a real park.

In [21]:
print(parks_raw["AREA_TYPE"].value_counts())
print(parks_raw["HECTARES"].describe())

AREA_TYPE
NATURAL FEATURES RESERVE - BUSHLAND RESERVE                                                2453
NATURE CONSERVATION RESERVE                                                                 842
NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT                                              836
REGIONAL PARK - NOT SCHEDULED UNDER NATIONAL PARKS ACT                                      669
HISTORIC RESERVE                                                                            590
NATURAL FEATURES RESERVE - STREAMSIDE RESERVE                                               385
NATURAL FEATURES RESERVE - WILDLIFE RESERVE (HUNTING)                                       360
STATE PARK - SCHEDULE 2B, NATIONAL PARKS ACT                                                253
COASTAL RESERVE                                                                             249
NATURE CONSERVATION RESERVE - FLORA AND FAUNA RESERVE                                       246
FOREST PARK                   

### Decide on the "OTHER" rows individually

"OTHER" mixes real, well-known parks (for example Albert Park, Werribee
Park) with infrastructure and features that are not parks (pump
stations, depots, a river, a shipwreck dive site). Rather than
exclude or keep the whole category, list every row so each one can be
judged, and keep only the ones confirmed as real parks.

In [22]:
pd.set_option("display.max_rows", None)
parks_raw[parks_raw["AREA_TYPE"] == "OTHER"][
    ["PRIMS_ID", "NAME", "HECTARES"]
].sort_values("HECTARES", ascending=False)

,PRIMS_ID,NAME,HECTARES
1163,5061,Snobs Creek Hatchery,236.43
4095,4050,Albert Park,218.28
1785,4086,Werribee Park,158.25
4749,4875,Ex-HMAS Canberra Recreation Reserve,142.98
211,5216,Wimmera Heritage River Proposed (addition),130.13
2199,5229,Victoria Park,119.83
3662,5077,Anglesea Heath Alcoa Agreement Area,118.46
2104,5022,Little Lake Boort Recreation Area Park,88.29
5428,5077,Anglesea Heath Alcoa Agreement Area,85.39
2395,5077,Anglesea Heath Alcoa Agreement Area,85.10


### Exclude non-park and non-visitable rows

Two exclusions are applied together. `EXCLUDED_AREA_TYPES` drops whole
categories that are underwater, infrastructure or access-restricted
(marine parks, port facilities, reservoirs, lighthouse reserves).
Separately, most of the 113 "OTHER" rows are real, named parks (Werribee
Park, Albert Park, Westgate Park and others), but a minority are
operational depots and offices, infrastructure, a generic "minor area"
management category, or sites the public doesn't visit for recreation —
these are excluded by name via `OTHER_ROWS_TO_EXCLUDE`.

In [23]:
EXCLUDED_AREA_TYPES = [
    "MARINE NATIONAL PARK - SCHEDULE 7, NATIONAL PARKS ACT",
    "MARINE SANCTUARY - SCHEDULE 8, NATIONAL PARKS ACT",
    "PORT & COASTAL FACILITY",
    "PV MANAGEMENT SERVICES AGREEMENT OTHER LAND",
    "RESERVOIR PARK",
    "LIGHTHOUSE RESERVE",
]

OTHER_ROWS_TO_EXCLUDE = [
    "Wail Parks Victoria Depot", "Anglesea Depot", "Rosebud Depot",
    "Parks Victoria Depot - Barkly Ave Park", "Lacey St Depot Sale",
    "San Remo Depot", "Natimuk Depot", "Rainbow Depot",
    "Rossmoyne Depot Colac", "Former Depot Red Cliffs", "Whitfield Depot",
    "Port Campbell Parks Victoria Depot", "Port Campbell PV Office",
    "Public Offices", "Port Campbell Sewerage Pump Station",
    "French Island Barge Landing (Saltmine Point)", "Patterson River",
    "Merri Creek Land (368 Mahoneys Rd)", "Kew Land, Molesworth St",
    "Ecologically Managed Minor Area Park", "Snobs Creek Hatchery",
    "Ex-HMAS Canberra Recreation Reserve",
    "Lerderderg Four Wheel Drive Training Facility and Rifle Range Complex",
    "Wimmera Heritage River Proposed (addition)",
]


In [24]:
is_other = parks_raw["AREA_TYPE"] == "OTHER"
is_excluded_other = is_other & parks_raw["NAME"].isin(OTHER_ROWS_TO_EXCLUDE)
is_excluded = (
    parks_raw["AREA_TYPE"].isin(EXCLUDED_AREA_TYPES) | is_excluded_other
)
print(f"Excluding {is_excluded.sum()} of {len(parks_raw)} parcels")

parks_raw = parks_raw[~is_excluded].reset_index(drop=True)

Excluding 159 of 8954 parcels


## Categorise into park types

Group `AREA_TYPE` into two categories: **major_park** (national, state,
wilderness and other parks under the National Parks Act, regional,
forest, metropolitan parks) and **local_reserve** (everything else that
survived filtering — historic, coastal and natural features/nature
conservation reserves, education areas).

In [25]:
MAJOR_PARK_AREA_TYPES = [
    "NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",
    "STATE PARK - SCHEDULE 2B, NATIONAL PARKS ACT",
    "WILDERNESS PARK - SCHEDULE 2A, NATIONAL PARKS ACT",
    "OTHER PARK - SCHEDULE 3, NATIONAL PARKS ACT",
    "OTHER PARK - SCHEDULE 9, NATIONAL PARKS ACT",
    "NATIONAL PARKS ACT SCHEDULE 4 PARK OR RESERVE",
    "PROPOSED NATIONAL PARKS ACT PARK OR PARK ADDITION",
    "REGIONAL PARK - NOT SCHEDULED UNDER NATIONAL PARKS ACT",
    "FOREST PARK",
    "METROPOLITAN PARK",
    "PHILLIP ISLAND NATURE PARK",
]

is_major_park = parks_raw["AREA_TYPE"].isin(MAJOR_PARK_AREA_TYPES)
parks_raw = parks_raw.assign(
    PARK_CATEGORY=np.where(is_major_park, "major_park", "local_reserve")
)

print(parks_raw["PARK_CATEGORY"].value_counts())

PARK_CATEGORY
local_reserve    6309
major_park       2486
Name: count, dtype: int64


## Convert to points

Take the centroid of each parcel's polygon as its representative point.
Since a park's parcels are kept separately, `PRIMS_ID` is no longer
unique, so `Facility ID` combines it with the row number to give each
parcel its own ID. `PRIMS_ID` and `NAME` are kept as separate columns,
so parcels belonging to the same park can still be grouped later if
needed.

A polygon's centroid can fall slightly outside an oddly shaped or
crescent-shaped parcel, but that matters little here since the routing
step snaps every point to the nearest road anyway.

In [26]:
parks_projected = parks_raw.to_crs(epsg=7855)
park_centroids = parks_projected.geometry.centroid.to_crs(epsg=4326)

parks = pd.DataFrame({
    "Facility ID": "PARK" + parks_raw.index.astype(str),
    "PRIMS_ID": parks_raw["PRIMS_ID"],
    "name": parks_raw["NAME"],
    "AREA_TYPE": parks_raw["AREA_TYPE"],
    "PARK_CATEGORY": parks_raw["PARK_CATEGORY"],
    "HECTARES": parks_raw["HECTARES"],
    "Latitude": park_centroids.y,
    "Longitude": park_centroids.x,
})

## Export PARKRES parks

Export the cleaned, categorised PARKRES parks (major_park and
local_reserve) to CSV. Combining with the OSM local_park extract is
deferred to a later step.

In [27]:
PARKRES_EXPORT_PATH = "../../external_data/parks_parkres.csv"
PARKRES_EXPORT_COLUMNS = [
    "Facility ID",
    "PRIMS_ID",
    "name",
    "AREA_TYPE",
    "PARK_CATEGORY",
    "HECTARES",
    "Latitude",
    "Longitude",
]

parks_export = parks[PARKRES_EXPORT_COLUMNS]

print(f"Facility ID unique: {parks_export['Facility ID'].is_unique}")
print(parks_export.isna().sum())
print(parks_export["PARK_CATEGORY"].value_counts())

parks_export.to_csv(PARKRES_EXPORT_PATH, index=False)
parks_export.head()

Facility ID unique: True
Facility ID      0
PRIMS_ID         0
name             0
AREA_TYPE        0
PARK_CATEGORY    0
HECTARES         0
Latitude         0
Longitude        0
dtype: int64
PARK_CATEGORY
local_reserve    6309
major_park       2486
Name: count, dtype: int64


,Facility ID,PRIMS_ID,name,AREA_TYPE,PARK_CATEGORY,HECTARES,Latitude,Longitude
0,PARK0,3369,Steiglitz Historic Park,HISTORIC RESERVE,local_reserve,429.11,-37.872803,144.173634
1,PARK1,3223,Glenmaggie Regional Park,REGIONAL PARK - NOT SCHEDULED UNDER NATIONAL P...,major_park,18.80,-37.928913,146.755517
2,PARK2,2924,Grant Historic Area,HISTORIC RESERVE,local_reserve,2.23,-37.343796,147.151107
3,PARK3,2924,Grant Historic Area,HISTORIC RESERVE,local_reserve,7401.34,-37.350306,147.130082
4,PARK4,4886,Warby-Ovens National Park,"NATIONAL PARK - SCHEDULE 2, NATIONAL PARKS ACT",major_park,3182.06,-36.220740,146.176289
